# Verifiable Quantum Circuit Compilation

## v5 · DFTHC-SOS Small-B Feasibility

Notebook 04 closed the cheap routes: with today's angle words the copy count cannot drop below `C = 40`, and no group
can be removed. Reading the scorer's own cost model (`scoreinfra/costs_prepare.py`, `SpinFreeData_V1`) gave a new lead.
`Q = 428` is mostly the **303 temporary qubits of the inner-state-preparation QROAM**, and the scorer picks that QROAM's
block size from `B`: it must satisfy `B ≥ 2^k`. With `B < 16` the block is forced small and `Q` falls to about 350.

| Shape `(R, B, C)` | Addresses | T (model) | Q (exact) | Chemistry gain at today's λ |
|---|---:|---:|---:|---:|
| (9, 143, 40), public | 1,363 | 16,233 | 428 | 1.00 |
| (24, 15, 15) | 436 | ≈ 6,723 | 354 | ≈ 2.9 |
| (36, 15, 10) | 616 | ≈ 7,977 | 355 | ≈ 2.5 |
| (45, 8, 8) | 436 | ≈ 6,677 | 339 | ≈ 3.1 |

State preparation and Givens costs are the scorer's exact formulas; the angle-loader Toffolis are scaled linearly with
the address count (rough). The open question is accuracy: a group of 15 vectors spans a tiny part of pair space.
**This notebook trains each small-B shape from scratch on CPU and reports the best continuous `error_2`.** A shape is
worth the next step (8-bit words and circuits) only if it gets well under ≈ 14 mHa, leaving room for word rounding.

A control run trains the public shape (9, 143, 40) from the same initialization for the same time. Lera's continuous
fit at that shape was ≈ 13.4 mHa; if our optimizer cannot approach that, a small-B failure says nothing.

**Kaggle setup:** Accelerator *None* (CPU only), Internet off. Inputs: the competition data and
[heliosli/qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live). Runtime ≈ 5.5 hours.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
SHAPES = [(9, 143, 40), (24, 15, 15), (36, 15, 10), (45, 8, 8)]   # control first
MINUTES = {(9, 143, 40): 60, (24, 15, 15): 85, (36, 15, 10): 80, (45, 8, 8): 80}
LBFGS_ITERS = 400                 # joint U/W iterations between exact W refits
W_SWEEPS = 6                      # block-coordinate sweeps of the exact W refit
SCALE = 1e4                       # objective scale for L-BFGS
LOG_EVERY = 90                    # seconds
TARGET_CONT = 0.014               # Ha: continuous error_2 that leaves room for 8-bit rounding
SEED = 20261102
RELEASES = {'heliosli/qcc-circuits-live': 'QCC_RELEASE_MANIFEST.json',
            'yoyobar/qcc-local-sota-20261031': 'portfolio-manifest.json'}

In [ ]:
# ── Session ───────────────────────────────────────────────────────────────────────────────────
import os, sys, glob, json, lzma, time, math, shutil, hashlib
from pathlib import Path

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
PUBLIC = ROOT / 'public_dfthcsos'
FITS = ROOT / 'fits'; FITS.mkdir(exist_ok=True)

def deep_find(name):
    return sorted(Path(p) for p in glob.glob(str(INPUT / '**' / name), recursive=True))

def find_competition():
    for p in (INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation',
              INPUT / 'google-verifiable-quantum-circuit-compilation'):
        if (p / 'verifiers').is_dir():
            return p
    return deep_find('verify_submission_invsqrt.py')[0].parent.parent.parent

def find_release():
    for name, marker in RELEASES.items():
        hits = deep_find(marker)
        if hits:
            return name, hits[0].parent
    raise FileNotFoundError('attach heliosli/qcc-circuits-live')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

COMP = find_competition()
RELEASE_DATASET, RELEASE = find_release()
sys.path[:0] = [str(COMP / 'kickmix'), str(COMP / 'verifiers')]
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from dfthcsos import verify as v_dfthcsos
from dfthcsos.scoreinfra.costs_prepare import SpinFreeData_V1
print('competition:', COMP, '| release:', RELEASE_DATASET, '| cores:', os.cpu_count())

if PUBLIC.exists():
    shutil.rmtree(PUBLIC)
PUBLIC.mkdir()
manifest = json.loads((RELEASE / RELEASES[RELEASE_DATASET]).read_text())
if RELEASE_DATASET.startswith('heliosli/'):
    item = next(i for i in manifest['files'] if i['file'] == 'dfthcsos/submission.npz')
    src, dst = RELEASE / item['stored_as'], PUBLIC / 'submission.npz'
    if item['stored_as'].endswith('.xz'):
        with lzma.open(src) as fi, open(dst, 'wb') as fo:
            shutil.copyfileobj(fi, fo)
    else:
        shutil.copyfile(src, dst)
    assert sha256(dst) == item['sha256']
else:
    shutil.copyfile(RELEASE / 'submission' / 'dfthcsos' / 'submission.npz', PUBLIC / 'submission.npz')

## Target and cost model

The two-body target `M` is rebuilt in packed pair space exactly as in notebooks 02 and 04 (checked there against the
official evaluator to 1e-7). The table recomputes the scorer's own state-preparation and Givens prices for every shape.

In [ ]:
INPUTS = v_dfthcsos.load_inputs(PUBLIC / 'submission.npz')
REF, SHIFT = INPUTS.ref, INPUTS.S
N = int(REF['orbitals'])
ii, jj = np.tril_indices(N)
d = np.sqrt(np.where(ii == jj, 1., 2.))
L = len(ii)
M = REF['tei'][ii[:, None], jj[:, None], ii[None, :], jj[None, :]].copy()
diag = (ii == jj).astype(float); shifted = SHIFT[ii, jj]
M += .5 * (shifted[:, None] * diag[None, :] + diag[:, None] * shifted[None, :])
M *= d[:, None] * d[None, :]
M = np.ascontiguousarray(.5 * (M + M.T))
NORM_M2 = float(np.sum(M * M))
EVAL, EVEC = np.linalg.eigh(M)
order = np.argsort(EVAL)[::-1]
EVAL, EVEC = EVAL[order], EVEC[:, order]
neg2 = float(np.sum(EVAL[EVAL < 0] ** 2))
def floor(k):
    return math.sqrt(neg2 + float(np.sum(EVAL[k:][EVAL[k:] > 0] ** 2)))

def shape_cost(R, B, C, ang_per_addr=9492 / 1363, ang_temp=77, ang_pers=40, brot=8):
    m = SpinFreeData_V1(R=R, B=B, C=C, N=N, chi=10, beta=brot)
    oq, iq = m.outer_state_prep_qubit_cost(), m.inner_state_prep_qubit_cost()
    T = m.outer_state_prep() + m.inner_state_prep() + m.givens_rotations() + ang_per_addr * (N + R * B)
    Q = oq['persistent'] + iq['persistent'] + ang_pers + max(oq['temporary'], iq['temporary'], ang_temp)
    return T, Q

T0, Q0 = shape_cost(9, 143, 40)
assert (round(T0), Q0) == (16233, 428), (T0, Q0)
COSTS = pd.DataFrame([dict(shape=str(s), addresses=N + s[0] * s[1], T_model=round(shape_cost(*s)[0]), Q=shape_cost(*s)[1],
                           gain_same_lambda=round(T0 * Q0 / (shape_cost(*s)[0] * shape_cost(*s)[1]), 3),
                           spectral_floor_mHa=round(1e3 * floor(s[0] * s[2]), 3)) for s in SHAPES])
print(COSTS.to_string(index=False))

## The fit

Each group `r` has `B` unit vectors `u_rb` (stored unnormalized) and weights `W_rbc`; copy `c` is the packed matrix
`a_rc = Σ_b W_rbc · pack(u_rb u_rbᵀ)` and the error is `‖F Fᵀ − M‖_F` with `F = [a_rc]`. Two moves alternate:

- **exact W refit**: for fixed vectors each group's best copies are the top positive eigenpairs of the residual
  projected on the span of its projectors (the block-coordinate update of notebook 04);
- **joint L-BFGS** on vectors and weights with analytic gradients, `∂f/∂F = 4(F Fᵀ − M)F`, pulled back through the
  packing and the normalization of `u`.

The fit ignores λ and the one-body error: it asks only whether the two-body limit is reachable at all.
Initialization: the top `R·C` eigenvectors of `M` (double-factorization leaves), dealt round-robin into groups;
each group's dictionary is a sign-invariant spherical k-means of its leaves' weighted orbital eigenvectors.

In [ ]:
def unit(V):
    return V / np.linalg.norm(V, axis=-1, keepdims=True)

def projectors(U):                      # (R, B, N) unit -> (R, B, L)
    return U[..., ii] * U[..., jj] * d

def copies(P, W):                       # -> F (L, R*C)
    R, B, C = W.shape
    return np.ascontiguousarray((W.transpose(0, 2, 1) @ P).transpose(2, 0, 1)).reshape(L, R * C)

def error2(F):
    MF = M @ F
    G = F.T @ F
    return math.sqrt(max(NORM_M2 - 2 * float(np.einsum('lk,lk->', F, MF)) + float(np.sum(G * G)), 0.))

def unpack(g):                          # (..., L) -> (..., N, N) symmetric with <S, Y> = <g, pack(Y)>
    S = np.zeros(g.shape[:-1] + (N, N))
    v = g / d
    S[..., ii, jj] = v
    S[..., jj, ii] = v
    return S

def objective(x, shape):
    R, B, C = shape
    V = x[:R * B * N].reshape(R, B, N)
    W = x[R * B * N:].reshape(R, B, C)
    nv = np.linalg.norm(V, axis=-1, keepdims=True)
    U = V / nv
    P = projectors(U)
    F = copies(P, W)
    MF = M @ F
    G = F.T @ F
    f = NORM_M2 - 2 * float(np.einsum('lk,lk->', F, MF)) + float(np.sum(G * G))
    GF = (4 * (F @ G - MF)).reshape(L, R, C)
    GFr = np.ascontiguousarray(GF.transpose(1, 0, 2))          # (R, L, C)
    gW = P @ GFr
    gP = W @ GFr.transpose(0, 2, 1)
    S = unpack(gP)
    gU = 2 * (S @ U[..., None])[..., 0]
    gV = (gU - U * np.sum(U * gU, axis=-1, keepdims=True)) / nv
    return SCALE * f, SCALE * np.concatenate([gV.ravel(), gW.ravel()])

def refit_w(U, W, sweeps=W_SWEEPS):
    """Exact block-coordinate W update for fixed vectors (top positive eigenpairs in each group's span)."""
    R, B, C = W.shape
    P = projectors(U)
    Q = [np.linalg.qr(P[r].T)[0] for r in range(R)]
    QMQ = [q.T @ (M @ q) for q in Q]
    Fr = [np.einsum('bc,bl->lc', W[r], P[r]) for r in range(R)]
    for _ in range(sweeps):
        for r in range(R):
            K = QMQ[r].copy()
            for s in range(R):
                if s != r:
                    t = Q[r].T @ Fr[s]
                    K -= t @ t.T
            val, vec = np.linalg.eigh(.5 * (K + K.T))
            top = np.argsort(val)[::-1][:C]
            lam = np.clip(val[top], 0, None)
            Fr[r] = (Q[r] @ vec[:, top]) * np.sqrt(lam)
    Wn = np.stack([np.linalg.lstsq(P[r].T, Fr[r], rcond=None)[0] for r in range(R)])
    return Wn

def init_dictionary(shape, rng):
    R, B, C = shape
    U = np.zeros((R, B, N))
    for r in range(R):
        leaves = list(range(r, R * C, R))
        cand, wts = [], []
        for k in leaves:
            Lk = unpack(EVEC[:, k][None, :])[0]                      # symmetric matrix of the packed leaf
            mu, vecs = np.linalg.eigh(Lk)
            keep = np.argsort(np.abs(mu))[::-1][:min(B, N)]
            cand.append(vecs[:, keep].T); wts.append(math.sqrt(max(EVAL[k], 0)) * np.abs(mu[keep]))
        X, w = np.concatenate(cand), np.concatenate(wts)
        centers = X[np.argsort(w)[::-1][:B]].copy()
        if len(centers) < B:
            centers = np.concatenate([centers, unit(rng.normal(size=(B - len(centers), N)))])
        for _ in range(20):
            lab = np.argmax(np.abs(X @ centers.T), axis=1)
            for b in range(B):
                sel = lab == b
                if sel.any():
                    S = (X[sel] * w[sel, None]).T @ X[sel]
                    centers[b] = np.linalg.eigh(S)[1][:, -1]
                else:
                    centers[b] = unit(rng.normal(size=N))
        U[r] = unit(centers)
    return U

## Gradient check

A finite-difference check of the analytic gradient on a small random shape before any long run.

In [ ]:
rng = np.random.default_rng(SEED)
shape = (3, 5, 4)
x = np.concatenate([rng.normal(size=3 * 5 * N), .01 * rng.normal(size=3 * 5 * 4)])
f0, g0 = objective(x, shape)
for t in range(3):
    direction = rng.normal(size=x.size); direction /= np.linalg.norm(direction)
    h = 1e-6
    fd = (objective(x + h * direction, shape)[0] - objective(x - h * direction, shape)[0]) / (2 * h)
    an = float(g0 @ direction)
    print(f'directional derivative: finite {fd:.8e} analytic {an:.8e}')
    assert abs(fd - an) <= 1e-4 * max(1., abs(an)), 'gradient check failed'
t0 = time.time(); objective(np.concatenate([rng.normal(size=24 * 15 * N), rng.normal(size=24 * 15 * 15)]), (24, 15, 15))
print(f'gradient check PASS | one evaluation at (24, 15, 15): {time.time() - t0:.2f} s')

In [ ]:
def train(shape, minutes):
    R, B, C = shape
    rng = np.random.default_rng(SEED)
    deadline = time.time() + 60 * minutes
    U = init_dictionary(shape, rng)
    W = refit_w(U, np.zeros((R, B, C)))
    err = error2(copies(projectors(U), W))
    best = dict(err=err, U=U.copy(), W=W.copy())
    curve = [(0., err)]
    t_start = time.time(); last_log = 0.
    print(f'{shape}: init error_2 {1e3 * err:.3f} mHa', flush=True)
    rounds = 0
    while time.time() < deadline - 5:
        rounds += 1
        x0 = np.concatenate([U.ravel(), W.ravel()])
        state = dict(x=x0)
        def cb(xk):
            state['x'] = xk.copy()
            if time.time() > deadline:
                raise StopIteration
        try:
            res = minimize(objective, x0, args=(shape,), jac=True, method='L-BFGS-B', callback=cb,
                           options=dict(maxiter=LBFGS_ITERS, maxcor=20, ftol=0., gtol=0., maxls=40))
            x = res.x
        except StopIteration:
            x = state['x']
        U = unit(x[:R * B * N].reshape(R, B, N)); W = x[R * B * N:].reshape(R, B, C)
        if time.time() < deadline - 30:
            W = refit_w(U, W)
        err = error2(copies(projectors(U), W))
        curve.append((round((time.time() - t_start) / 60, 2), err))
        if err < best['err']:
            best = dict(err=err, U=U.copy(), W=W.copy())
        if time.time() - last_log > LOG_EVERY:
            last_log = time.time()
            print(f'  {shape} round {rounds} | {curve[-1][0]:6.1f} min | error_2 {1e3 * err:8.3f} mHa | best {1e3 * best["err"]:8.3f}', flush=True)
    np.savez(FITS / f'fit_{R}_{B}_{C}.npz', U=best['U'], W=best['W'])
    return best['err'], curve, rounds

RESULTS = {}
for shape in SHAPES:
    t0 = time.time()
    err, curve, rounds = train(shape, MINUTES[shape])
    RESULTS[str(shape)] = dict(error_2_mHa=1e3 * err, minutes=round((time.time() - t0) / 60, 1), rounds=rounds,
                               curve=[(t, round(1e3 * e, 4)) for t, e in curve])
    print(f'==> {shape}: best continuous error_2 {1e3 * err:.3f} mHa in {RESULTS[str(shape)]["minutes"]} min, {rounds} rounds', flush=True)
    (ROOT / 'small_b.json').write_text(json.dumps(RESULTS, indent=1))

## Verdict

Continuous `error_2` only; the next step for a passing shape would add λ and the one-body error to the objective,
then 8-bit words and circuits. The curve tail shows whether a shape was still improving when its time ran out.

In [ ]:
rows = []
control = RESULTS[str(SHAPES[0])]['error_2_mHa']
for s in SHAPES:
    r = RESULTS[str(s)]
    tail = [e for t, e in r['curve'] if t >= .8 * r['minutes']]
    rows.append(dict(shape=str(s), best_error_2_mHa=round(r['error_2_mHa'], 3),
                     still_improving=bool(len(tail) > 1 and tail[0] - tail[-1] > .01 * tail[0]),
                     under_target=r['error_2_mHa'] < 1e3 * TARGET_CONT, under_limit=r['error_2_mHa'] < 20.,
                     gain_same_lambda=float(COSTS.set_index('shape').loc[str(s), 'gain_same_lambda']),
                     spectral_floor_mHa=float(COSTS.set_index('shape').loc[str(s), 'spectral_floor_mHa'])))
VERDICT = pd.DataFrame(rows)
print(VERDICT.to_string(index=False))
print(f'\ncontrol (9, 143, 40) reached {control:.3f} mHa; Lera\'s continuous fit there was about 13.4 mHa')
ok = VERDICT[(VERDICT['shape'] != str(SHAPES[0])) & VERDICT.under_target]
print('small-B shapes worth the next step:', list(ok['shape']) or 'none')
(ROOT / 'small_b.json').write_text(json.dumps(dict(results=RESULTS, verdict=rows, costs=COSTS.to_dict('records'),
                                                   minutes=round((time.time() - NB_T0) / 60, 1)), indent=1, default=float))
print(f'SMALL-B SCREEN DONE | {(time.time() - NB_T0) / 60:.1f} minutes')

# Sources and credits

- **Lera (marculera)**: the public `(9, 143, 40)` shape and its continuous fit (≈ 13.4 mHa) used as the control
  reference: [DF-THC 10x: Compact Groups From the Hamiltonian](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian).
  Lera credits Michael Rustad for the B ≥ 129 workspace cliff of the angle loaders; this notebook looks at the opposite
  end of `B`, where the state-preparation QROAM is forced small.
- **Dmitrii Gluzdov**: the Toffoli breakdown behind the per-address angle cost
  ([Quantum Circuit Compilation from Source](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source)).
- **heliosli**: the public release used for the reference factorization and fixed shift
  ([qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live)).
- **Tanuj Khattar, Nicholas C. Rubin, Craig Gidney, Michael Broughton, Guang Hao Low and Ryan Babbush**: the verifier and
  its `SpinFreeData_V1` cost model ([arXiv:2610.23832](https://arxiv.org/abs/2610.23832)); **Guang Hao Low et al.**,
  DFTHC ([arXiv:2502.15882](https://arxiv.org/abs/2502.15882)); **Joonho Lee et al.**, THC
  ([arXiv:2011.03494](https://arxiv.org/abs/2011.03494)) for the leaf-and-dictionary picture.
- Ours: the small-B lead from the scorer's QROAM rule, the fit and the control design.